# **Bi-Directional Occupancy Intelligence System**

## The Goal
The aim of this project is to build a smart camera system that automatically tracks how many people enter and leave a building. Instead of just seeing a crowd, the system understands direction, helping managers know exactly how many people are inside at any given moment.

---

## Problem Statement & Resolution
* **The Manual Burden:** Counting people by hand is slow, expensive, and prone to human error.
* **The "Direction" Challenge:** Standard cameras can see a person, but they don't know if that person is coming or going.
* **The Solution:** We use **AI "memory."** The system gives every person a unique ID and watches their path. If they cross a digital line moving forward, they are counted as **"In"**; if they move backward, they are counted as **"Out."**

---

## Key Features
* **Real-Time Detection:** Recognizes people instantly as they appear on screen.
* **Directional Counting:** Automatically sorts foot traffic into "In" and "Out" categories.
* **Smart Tracking:** Keeps track of the same person even if they are briefly blocked by a door or another person.
* **Live Occupancy:** Provides a live total of how many people are currently in the room.

---

## Tools Used
| Tool | Purpose |
| :--- | :--- |
| **Python** | The main programming language and logic of the project. |
| **YOLOv8** | A powerful AI model famous for recognizing objects at high speeds. |
| **OpenCV** | The library that helps the computer process video frames and draw counting lines. |

# **IMPORTING LIBRARIES**

This section loads all required libraries for data handling, visualization, and model training.

In [35]:
# Import Libraries

import os
import cv2
import yaml
import csv
import random
import shutil
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

# **LOADING THE DATASET**

## Loading The Dataset

We inspect the dataset structure to ensure images and labels are correctly organized for YOLO training.

In [36]:
print(os.getcwd())

/home/kagendo/Desktop/YOLO_People_Counting_System/training


In [37]:
DATASET_PATH = r"/home/kagendo/Desktop/YOLO_People_Counting_System/training/data"

for root, dirs, files in os.walk(DATASET_PATH):
    print(f"{root} -> {len(files)} files")
print_tree(DATASET_PATH)

/home/kagendo/Desktop/YOLO_People_Counting_System/training/data -> 0 files
/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/train -> 0 files
/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/train/data -> 15211 files
/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/valid -> 0 files
/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/valid/valid -> 1432 files
data/
    train/
        data/
            000108_jpg.rf.cf69a1e17689a2a7998255f0ab592f31.jpg
            fourth_034_png_jpg.rf.e83d391edfbe2f04d7dd1dd9ffdf2887.jpg
            2008_001504_jpg.rf.c5ee5c4e34a920979ba3272930739c5b.jpg
            fourth_032_png_jpg.rf.5fad28d971436a122425dcf4bdaa8186.jpg
            2008_006294_jpg.rf.85445c3a7d2e3c4cb2a9a77f17f17229.jpg
            131_person_jpg.rf.375225e68b157b1399ba0fb428122ea9.jpg
            000048_jpg.rf.dfcc0e6224df6b6da315e531bb1e1f41.jpg
            2008_008148_jpg.rf.de896ce533be3b79a544401f4ba9a683.jpg
            2

### From the above output;

**Data Distribution**
* **Training Set:** 15,211 files
* **Validation Set:** 1,432 files

**Structural Observation**
The dataset uses a **nested folder structure**. The primary `train` and `valid` directories are empty; all images are located in the sub-directories:
* `.../train/data/`
* `.../valid/valid/`

**Action Required**
The `data.yaml` configuration must point directly to these specific sub-folders to prevent training errors.

# **DATA EXPLORATION**

In [43]:
df = pd.read_csv("/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/train/data/_annotations.csv")
print(df.head())
print(df.info())
print(df.describe())
print(f"Total annotations: {len(df)}")

                                            filename  width  height   class  \
0  2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...    500     375  person   
1  2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...    500     375  person   
2  2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...    500     375  person   
3  004574_jpg.rf.7c8cea69d7be45f58febcede26ef0c6e...    500     333  person   
4  004574_jpg.rf.7c8cea69d7be45f58febcede26ef0c6e...    500     333  person   

   xmin  ymin  xmax  ymax  
0   219    98   269   283  
1   114   124   155   263  
2    43   139    98   340  
3   145   118   229   333  
4   285   105   349   329  
<class 'pandas.DataFrame'>
RangeIndex: 100082 entries, 0 to 100081
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   filename  100082 non-null  str  
 1   width     100082 non-null  int64
 2   height    100082 non-null  int64
 3   class     100082 non-null  str  
 4   xmin      100082 non-null

In [44]:
df = pd.read_csv("/home/kagendo/Desktop/YOLO_People_Counting_System/training/data/valid/valid/_annotations.csv")
print(df.head())
print(df.info())
print(df.describe())
print(f"Total annotations: {len(df)}")

                                            filename  width  height   class  \
0  BSMT1XOO121P_jpg.rf.3172b37162bba43942f1a82395...    500     375  person   
1  point03_140_png_jpg.rf.4166a3ba6f32cf3fad662d2...   1280     720  person   
2  point03_140_png_jpg.rf.4166a3ba6f32cf3fad662d2...   1280     720  person   
3  point01_179_png_jpg.rf.4b397a9403192a0c286c5f3...   1280     720  person   
4  2010_003325_jpg.rf.9caa17b2dd1dade077477d4ae80...    500     375  person   

   xmin  ymin  xmax  ymax  
0   254   136   350   304  
1   406   138   609   488  
2   681     0   711   101  
3   836   244   976   579  
4   213    40   476   316  
<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   filename  10683 non-null  str  
 1   width     10683 non-null  int64
 2   height    10683 non-null  int64
 3   class     10683 non-null  str  
 4   xmin      10683 non-null  int64
 

# **CLEANING THE DATA**

# **DATA PREPARATION**

# **MODEL SELECTION**

# **MODELTRAINING/FINE TUNING**

# **HYPERPARAMETER TUNING**

# **MODEL SAVING**

# **MODEL EVALUTION**

# **INFERENCE**

# **PEOPLE COUNTING LOGIC**

# **VISUALIZATION**

# **DEPLOYMENT**